In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("Inicjalizacja Projektu: Klasyfikacja ryzyka opóźnienia dostawy")

# Wczytywanie danych - Github
dataset = pd.read_csv(r'C:\Users\Alicja\Desktop\kurs-datascience\DataCoSupplyChainDataset.csv', encoding='latin1')
dataset.head()

# Definiowanie zmiennych 
y = dataset['Late_delivery_risk']

dane = [
    'Days for shipment (scheduled)', # Planowany czas dostawy
    'Benefit per order',             # Zysk na zamówieniu
    'Sales per customer',            # Sprzedaż na klienta
    'Product Price'                  # Cena produktu
]
X = dataset[dane]

# Wypełnienie braków danych za pomocą mediany
X = X.fillna(X.median())

# Podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standaryzacja
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Słownik  na wyniki
precision_results = {}
recall_results = {}

# Definicja modeli
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=200, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight='balanced')
}

for name, model in models.items():
    print(f"\n Trenowanie modelu: {name}")
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(classification_report(y_test, y_pred))

    # Zapis wyników do wykresu
    precision_results[name] = precision_score(y_test, y_pred, pos_label=1)
    recall_results[name] = recall_score(y_test, y_pred, pos_label=1)

# MLP
mlp = MLPClassifier(
hidden_layer_sizes=(10, 5), 
activation='relu', 
solver='adam', 
max_iter=500, 
random_state=42,
early_stopping=True,
validation_fraction=0.15       
)

# Właczenie stopera
start = time.time()

mlp.fit(X_train_scaled, y_train)
train_time = time.time() - start # Wyłączenie stoperu
y_pred_mlp = mlp.predict(X_test_scaled)

print("\n Wynik dla MLP")
print(f"Czas treningu sieci: {train_time:.3f} sekund")
print(f"Liczba wykorzystanych epok: {mlp.n_iter_} (z maksymalnie 500)")
print(f"Architektura sieci: {X_train.shape[1]} wejścia --> {mlp.hidden_layer_sizes} --> 1 wyjście")

print("\n Raport końcowy dla MLP")
print(classification_report(y_test, y_pred_mlp))

# Zapis wyników MLP do wykresu
precision_results['MLP Network'] = precision_score(y_test, y_pred_mlp, pos_label=1)
recall_results['MLP Network'] = recall_score(y_test, y_pred_mlp, pos_label=1)

print("\nWszystkie modele gotowe! Generuję wykresy...")


# GENEROWANIE WYKRESÓW
# WYKRES 1: Krzywa błędu walidacji dla MLP (Dostosowana do early_stopping)
plt.figure(figsize=(8, 4))
# Obliczamy błąd jako: 1 - dokładność walidacji
validation_loss = [1 - score for score in mlp.validation_scores_]
plt.plot(validation_loss, color='red', linewidth=2, label='Błąd Walidacji (Validation Loss)')
plt.title('Krzywa uczenia sieci MLP (Jak malał błąd na zbiorze walidacyjnym)')
plt.xlabel('Epoka (Lekcja)')
plt.ylabel('Wartość błędu')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# WYKRES 2: Porównanie Precision i Recall dla klasy 1 (Zostaje bez zmian)
model_names = list(precision_results.keys())
precisions = list(precision_results.values())
recalls = list(recall_results.values())

x_axis = np.arange(len(model_names))

plt.figure(figsize=(10, 5))
plt.bar(x_axis - 0.2, precisions, 0.4, label='Precision (Precyzja)', color='skyblue', edgecolor='black')
plt.bar(x_axis + 0.2, recalls, 0.4, label='Recall (Czułość)', color='salmon', edgecolor='black')

plt.xticks(x_axis, model_names, rotation=15)
plt.title('Porównanie skuteczności modeli dla klasy 1 (Spóźnienia)')
plt.ylabel('Wartość metryki (Od 0 do 1)')
plt.ylim(0, 1.1)  
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()
